In [1]:
%load_ext autoreload
import sys
from datetime import datetime
from ipywidgets import widgets
from matplotlib.lines import Line2D
from ptflops import get_model_complexity_info
from torch.optim import Adam
import torch
from torch import nn

sys.path.append("..")
%autoreload 2
from src import *

plt.rcParams.update({'font.size': 20})

In [2]:
df_x, df_y = read_dataset(stage="original")
# df_x['np/ng'] = df_x['np'] / df_x['ng']
df_x.drop(inplace=True, columns=["ias", "oat"])
df_x["trq_margin"] = df_y["trq_margin"]
df_x["task"] = (df_x["np"] / df_x["ng"] < 1).astype(int) + 1

In [3]:
def get_title(model,  epochs, times, lr, is_file=False):
    additional = f'grid={model.grid_size}' if type(model) in [PyKAN, EfficientKAN] else ''
    return f'{model.__class__.__name__}_layers=[{"-".join(map(str, model.layers))}]_epochs={epochs}_times={times}_lr={lr}_{additional}_{datetime.now().ctime().replace(":", "-")}' if is_file else f'{model.__class__.__name__} Epochs={epochs} Times={times} Lr={lr}'

In [4]:
def plot(
    losses: tuple[list, list, list],
    tasks: list[int],
    train=False,
    valid=False,
    title=None,
    scale=1,
    ax=None,
    standalone=True,
    label=None,
    metric="test_loss",
    ylim=None,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(20 * scale, 10 * scale))
        ax.set_xticks(range(0, len(losses[0]) * len(losses[0][0]) + 1, len(losses[0][0])))
        for i, test_loss in enumerate(losses[0]):
            ax.add_line(
                Line2D([(i + 1) * len(test_loss)] * 2, [0, 99], linestyle="--", color="black", linewidth=1, alpha=0.5)
            )
            ax.fill_between(
                range(i * len(test_loss), (i + 1) * len(test_loss) + 1),
                -99,
                99,
                alpha=0.25,
                color="tab:green" if i % 2 == 0 else "tab:red",
                label=None if i > 1 else "np > ng" if i % 2 == 0 else "np < ng",
            )
        ax.set_ylim(ylim if ylim else [-1, 0] if min(losses[0][-1]) < 0 else [0, 2])
        ax.set_xlabel("Epoch")
        ax.set_ylabel(" ".join(map(lambda s: s.capitalize(), metric.split("_"))))
    if train:
        ax.plot([x for xs in losses[0] for x in xs], color="tab:orange", label="train")
    if valid:
        ax.plot([x for xs in losses[1] for x in xs], color="tab:green", label="valid")

    ax.plot(
        [
            sum(x[t][metric] * x[t]["samples"] for t in tasks) / sum(x[t]["samples"] for t in tasks)
            for xs in losses[2]
            for x in xs
        ],
        color="tab:blue" if standalone else None,
        label="test" if label is None else label,
        linewidth=3,
    )

    if title:
        ax.set_title(title)
    if standalone:
        ax.legend()
        plt.show()
    return ax

In [5]:
def visualizer_gui(tasks, metric="test_loss", ylim=None, title=None, loc="best"):
    output = widgets.Output()
    selected_files = []
    buttons = {}
    losses = {}
    last_ax = {"ax": None}

    def select_file(file, deselect_all=False):
        with output:
            try:
                output.clear_output()
                if deselect_all:
                    selected_files.clear()
                else:
                    if file in selected_files:
                        selected_files.remove(file)
                    else:
                        selected_files.append(file)

                # Set button styles
                for f, button in buttons.items():
                    button.button_style = "primary" if f in selected_files else ""

                # Plot
                ax = None

                def format_label(file):
                    pieces = file.replace("_layers=", "").replace("EfficientKAN", "KAN").split("_")[:-1]
                    pieces = filter(
                        lambda piece: not any(x in piece for x in ["epoch", "times"]) and len(piece) > 1, pieces
                    )
                    return ", ".join(pieces)

                for selected_file in selected_files:
                    ax = plot(
                        losses[selected_file],
                        tasks,
                        title=title if title else file,
                        train=False,
                        ax=ax,
                        standalone=False,
                        label=format_label(selected_file),
                        metric=metric,
                        ylim=ylim,
                    )
                if ax:
                    ax.legend(loc=loc)
                    last_ax["ax"] = ax
                    plt.show()
                save.layout.display = "block" if ax else "none"
            except Exception as e:
                print(e)

    def save_fig():
        if last_ax["ax"] is not None:
            fig = last_ax["ax"].get_figure()
            filename = f"{last_ax['ax'].get_title() or 'plot'}.png"
            fig.savefig(f"img/{filename}", dpi=300, bbox_inches="tight")

    files = os.listdir("results/itl/")
    files = sorted(files, key=lambda f: os.path.getctime(os.path.join("results/itl/", f)), reverse=False)

    for file in files:
        if os.path.isdir(f"results/itl/{file}"):
            continue
        with open(f"results/itl/{file}", "rb") as f:
            losses[file] = pickle.load(f)

        btn = widgets.Button(
            description=" ——— ".join(file.split("_")[:-1]),
            layout=widgets.Layout(width="auto"),
            style={"font_size": "20px", "text_decoration": "underline" if min(losses[file][0][-1]) < 0 else ""},
            button_style="",
        )
        btn.on_click(lambda _, f=file: select_file(f))
        buttons[file] = btn
    clear = widgets.Button(
        description="✖️ Clear",
        layout=widgets.Layout(width="auto"),
        style={"font_size": "20px", "font_weight": ""},
        button_style="warning",
    )
    clear.on_click(lambda _: select_file(None, deselect_all=True))
    save = widgets.Button(
        description="💾 Save Plot",
        layout=widgets.Layout(width="auto"),
        style={"font_size": "20px", "font_weight": ""},
        button_style="success",
    )
    save.layout.display = "none"
    save.on_click(lambda _: save_fig())
    spacer = widgets.HTML(value="<div style='margin-top:30px;'></div>")
    display(widgets.VBox([*buttons.values(), clear, spacer, output, save]))

In [6]:
def get_criterion(task: int):
    def criterion(x: torch.Tensor, y: torch.Tensor):
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2]).to(x.device))(x[:, task - 1], y.squeeze())

    return criterion


def itl_train(
    model: PHMNetwork,
    tasks: list[int],
    epochs: int,
    times: int,
    lr=1e-3,
    device="cpu",
    silent=True,
) -> tuple[list, list, list]:
    model.reset()
    train_losses, valid_losses, test_metrics = [], [], []
    optimizer = Adam(model.parameters(), lr=lr)
    trainset, validset, testset, _ = split_dataset(
        df_x,
        df_y["faulty"],
        0.5,
        standardize_y=False,
        group_by="task",
        group_testset=True,
        device=device,
        seed=0,
    )
    for i, task in enumerate(tqdm(tasks * times)):
        min_length = min(trainset[1][0].shape[0], trainset[2][0].shape[0])
        for t in tasks:
            trainset[t] = (trainset[t][0][:min_length], trainset[t][1][:min_length])

        def test():
            metrics = {}
            for t in tasks:
                metrics[t] = model.test(
                    testset[t],
                    batch_size=2048,
                    silent=silent,
                    task=t,
                )
                metrics[t]["samples"]=len(testset[t])
                
            test_metrics[i].append(metrics)

        test_metrics.append([])
        train_loss, valid_loss = model.fit(
            trainset[task],
            validset[task],
            optimizer,
            epochs=epochs,
            batch_size=2048,
            callback=test,
            silent=silent,
            criterion=get_criterion(task=task),
        )
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        # model.save(f'IDL_{model.__class__.__name__}_{domain}')

    # Plot graph
    if not silent:
        plot(
            (train_losses, valid_losses, test_metrics),
            title=get_title(model, epochs, times, lr),
            metric="avg_test_score",
            tasks=tasks
        )

    # Save to file
    with open(f"results/itl/{get_title(model, epochs, times, lr, is_file=True)}", "wb") as f:
        pickle.dump((train_losses, valid_losses, test_metrics), f)
    return train_losses, valid_losses, test_metrics

In [23]:
TASKS = [1, 2]
EPOCHS = 5
TIMES = 1
LR = 50e-2

## Finding a matching shape for KAN and MLP with `ptflops`

In [14]:
effKAN = EfficientKAN([len(df_x.columns) - 1, 51, 51, len(TASKS)], "classification", grid_size=20, continual_learning=True)
mlp = MLP([len(df_x.columns) - 1, 256, 256, len(TASKS)], "classification")

for net in [effKAN, mlp]:
    flops, params = get_model_complexity_info(
        net, (len(df_x.columns) - 1,), as_strings=True, print_per_layer_stat=False
    )
    print(f"[{net.__class__.__name__}] FLOPs: {flops} | Params: {params}")

[EfficientKAN] FLOPs: 108 Mac | Params: 69.21 k
[MLP] FLOPs: 68.1 KMac | Params: 68.1 k


# KAN

In [20]:
effKAN = EfficientKAN(
    [len(df_x.columns) - 1, 51, 51, len(TASKS)],
    "classification",
    grid_size=20,
    continual_learning=True,
    device="cuda",
)
itl_train(effKAN, TASKS, epochs=EPOCHS, times=TIMES, lr=LR, device="cuda")
pass

100%|██████████| 2/2 [00:18<00:00,  9.44s/it]


# MLP

In [24]:
mlp = MLP(
    [len(df_x.columns) - 1, 256, 256, len(TASKS)],
    "classification",
    device="cuda",
)
itl_train(mlp, TASKS, epochs=EPOCHS, times=TIMES, lr=LR, device="cuda")
pass

  0%|          | 0/2 [00:00<?, ?it/s]/home/vmorelli-iit.local/Projects/PHM_North_America_2024_Challenge/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vmorelli-iit.local/Projects/PHM_North_America_2024_Challenge/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vmorelli-iit.local/Projects/PHM_North_America_2024_Challenge/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

# Visualize the results with the visualizer GUI

In [26]:
TITLES = [
    "Comparison of different KAN grid sizes",
    "Comparison of different MLP architecture sizes",
    "MLP vs KAN without Replay",
    "MLP vs KAN with Replay",
]
visualizer_gui(TASKS, metric="avg_test_score", ylim=[0, 1], title=TITLES[3])